# Scale run: DCTR + FLD, then CNN

One session, two parts, run back to back:

- **Part A - DCTR + FLD ensemble** (about 1-2 h). The feature-based detector,
  ported from MATLAB to Python so it can run here.
- **Part B - EfficientNet-B0** (about 5 h on two T4s). Same code as the
  300-image run, with more data.

**Before you start:** *Accelerator -> GPU T4 x2*, *Internet -> On*, and
*+ Add Input ->* the `stego-eval-2k` dataset. Then **Save Version -> Save & Run
All** and close the tab.

Part A failing does not stop Part B, and the CNN stops itself cleanly before
the 12-hour cap so nothing finished is lost. The last cell prints both
detectors side by side and writes `scale_results.zip`.

**Reading P_E:** chance is not 0.50. Each part runs its own cover-vs-cover null
test and reports results against that.

In [ ]:
# ---- Cell 1: preflight -------------------------------------------------
import json
import pathlib
import subprocess
import sys
import time

import torch

NOTEBOOK_T0 = time.time()
SESSION_LIMIT_MIN = 11.2 * 60      # Kaggle kills the session at 12 h

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
if N_GPU == 0:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")
for i in range(N_GPU):
    print("  GPU", i, torch.cuda.get_device_name(i))

for pkg in ("timm", "jpeglib"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    print(pkg, __import__(pkg).__version__)

# A dataset shipped as ONE archive file is unpacked here, onto the session's
# local disk. Tens of thousands of loose files in a Kaggle dataset can stall a
# session before the first cell runs; a single file does not, and training then
# reads from local disk instead of the input mount.
import zipfile

LOCAL = pathlib.Path("/kaggle/temp" if pathlib.Path("/kaggle/temp").is_dir() else "/tmp") / "stego_data"
print("looking for data under /kaggle/input ...", flush=True)
packed = [p for ext in ("*.bin", "*.zip")
          for p in sorted(pathlib.Path("/kaggle/input").rglob(ext))
          if zipfile.is_zipfile(p)]
for z in packed:
    t_z = time.time()
    dest = LOCAL / z.stem
    print("unpacking", z, "(%.2f GB) ->" % (z.stat().st_size / 1e9), dest, flush=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print("  done in %.0f s" % (time.time() - t_z), flush=True)

# If several stego-eval datasets are attached, take the largest.
found = []
roots = [pathlib.Path("/kaggle/input")] + ([LOCAL] if LOCAL.is_dir() else [])
for root in roots:
    for m in root.rglob("MANIFEST.json"):
        found.append((json.loads(m.read_text())["cover_count"], m.parent))
if not found:
    raise SystemExit("No MANIFEST.json under /kaggle/input - add the dataset "
                     "via '+ Add Input'.")
found.sort(key=lambda t: t[0])
N_COVERS, DATA = found[-1]
print("data root:", DATA, "|", N_COVERS, "covers")
mf = json.loads((DATA / "MANIFEST.json").read_text())
for c in mf["cells"]:
    print("  " + c["cell"].ljust(18) + str(c["count"]).rjust(6) + " images   " + c["channel"])

OUT = pathlib.Path("/kaggle/working/scale_results")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile m4.py
"""M4 CNN steganalysis - reference implementation.

Paste into a Kaggle notebook (GPU on) or run as a script. Written to be correct
first and fast second; every choice that looks unusual is deliberate and is
explained in 02_PITFALLS.md.

Usage (Kaggle: attach the dataset first):

    python kaggle_notebook.py --data /kaggle/input/stego-eval \
        --out /kaggle/working/m4_results --null-test
    python kaggle_notebook.py --data /kaggle/input/stego-eval \
        --out /kaggle/working/m4_results --cells GMAS_0.1
    python kaggle_notebook.py --data /kaggle/input/stego-eval \
        --out /kaggle/working/m4_results --all

Requires: torch, timm, numpy, pillow.
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import pathlib
import random
import shutil
import tempfile
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

SEEDS = [0, 1, 2, 3, 4]
# Early stopping on VALIDATION only: stop a seed after this many epochs with no
# strict improvement in validation P_E. None = always run every epoch.
PATIENCE = None


# ---------------------------------------------------------------- metrics ---
def p_error(sc: np.ndarray, ss: np.ndarray) -> float:
    """min over thresholds of 0.5*(P_FA + P_MD).

    Higher score must mean 'more likely stego'.

    WARNING - inverted scores do NOT show up as a value near 1.0. Because this
    is a min over thresholds, the degenerate threshold (call everything cover)
    always achieves 0.5, so backwards wiring collapses to ~0.5: it reads as
    'perfectly secure' and is indistinguishable from a passing null test. The
    only way to see it is to score the other direction as well - run_cell does,
    and reports p_e_reversed. If reversed is clearly LOWER, the scores are
    upside down.
    """
    best = 1.0
    for t in np.unique(np.concatenate([sc, ss])):
        best = min(best, 0.5 * ((sc >= t).mean() + (ss < t).mean()))
    return float(best)


def null_expectation(n_test_pairs: int) -> tuple[float, float]:
    """(mean, sd) of P_E when there is NO signal, at this test-set size.

    P_E = 0.5 * (1 - D+), where D+ is the one-sided two-sample
    Kolmogorov-Smirnov statistic. D+ is not zero under the null - it shrinks
    like 1/sqrt(n) - so a perfect pipeline reads BELOW 0.5: about 0.453 at 75
    test pairs, 0.466 at 150, 0.476 at 300. "Chance" is this number, not 0.50,
    and a P_E at or above it means "not detected at this sample size".

    Constants checked by simulation, 4000 trials per n.
    """
    n = max(int(n_test_pairs), 1)
    mean = 0.5 * (1.0 - math.sqrt(math.pi / 8.0) / math.sqrt(n / 2.0))
    return mean, 0.232 / math.sqrt(n)


def md_at_fa(sc: np.ndarray, ss: np.ndarray, target_fa: float = 0.01) -> float:
    """Missed detection at a fixed false-alarm rate - a realistic warden."""
    return float((ss < np.quantile(sc, 1 - target_fa)).mean())


# ---------------------------------------------------------------- dataset ---
class PairedSet(Dataset):
    """Cover/stego pairs for one cell.

    Loads JPEGs to float32 in [0,1] at native 512x512 and NEVER re-encodes.

    Augmentation is limited to flips and 90-degree rotations: these are the only
    transforms that permute pixels without resampling, so they preserve the
    embedding signal. Anything involving interpolation, cropping or
    re-compression destroys what we are trying to detect.
    """

    def __init__(self, cover_dir, stego_dir, stems, train: bool):
        self.cover_dir = pathlib.Path(cover_dir)
        self.stego_dir = pathlib.Path(stego_dir)
        self.stems = list(stems)
        self.train = train

    def __len__(self) -> int:
        return len(self.stems) * 2

    def __getitem__(self, i):
        stem = self.stems[i // 2]
        is_stego = i % 2
        d = self.stego_dir if is_stego else self.cover_dir
        x = np.asarray(Image.open(d / f"{stem}.jpg"), dtype=np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                x = np.fliplr(x)
            if random.random() < 0.5:
                x = np.flipud(x)
            k = random.randint(0, 3)
            if k:
                x = np.rot90(x, k)
        x = np.ascontiguousarray(x)[None]  # (1, H, W)
        return torch.from_numpy(x), torch.tensor(float(is_stego)), stem


# ------------------------------------------------------------------ model ---
def build_model() -> nn.Module:
    """EfficientNet-B0, ImageNet weights, single-channel input.

    No resize: the network is fully convolutional and accepts 512x512 directly.
    Downsampling to 224 would average away the residual the detector needs.
    `in_chans=1` lets timm adapt the stem rather than us faking a 3-channel
    image. Inputs are scaled to [0,1] only - no ImageNet mean/std, which is
    meaningless on a replicated grayscale channel.
    """
    import timm

    return timm.create_model("efficientnet_b0", pretrained=True,
                             num_classes=1, in_chans=1)


# ---------------------------------------------------------------- scoring ---
def score_loader(model, dl, device, use_amp):
    """Score every image in a loader -> (cover scores, stego scores, rows)."""
    model.eval()
    sc, ss, keep = [], [], []
    with torch.no_grad():
        for x, y, stem in dl:
            with torch.cuda.amp.autocast(enabled=use_amp):
                o = model(x.to(device)).squeeze(1).float().cpu().numpy()
            for oi, yi, si in zip(o, y.numpy(), stem):
                (ss if yi > 0.5 else sc).append(oi)
                keep.append((si, "stego" if yi > 0.5 else "cover", float(oi)))
    return np.asarray(sc), np.asarray(ss), keep


# --------------------------------------------------------------- one cell ---
def run_cell(cover_dir, stego_dir, stems, device, epochs, bs, lr, out_scores):
    per_seed_pe, per_seed_md, best_epochs = [], [], []
    per_seed_rev, per_seed_val = [], []
    epochs_run = []
    rows = []
    n_tr = n_va = n_te = 0

    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # PAIRED split: the index decides, so cover i and stego i stay together.
        # Splitting the two classes independently would put a cover in training
        # and its own stego in test - the network then recognises the photograph
        # instead of the embedding. See 02_PITFALLS.md section 1.
        #
        # Three-way: 40% train / 10% validation / 50% test. The validation
        # slice exists only to choose the epoch. Choosing it on the test set
        # instead - the common shortcut - takes a min over ~40 noisy estimates
        # and drags P_E down by ~0.07 with no signal present at all, which both
        # fails the null test on a correct pipeline and understates the
        # security of every genuinely hard-to-detect method.
        idx = np.random.default_rng(seed).permutation(len(stems))
        half = len(stems) // 2
        n_val = max(2, int(round(0.2 * half)))
        va = [stems[i] for i in idx[:n_val]]
        tr = [stems[i] for i in idx[n_val:half]]
        te = [stems[i] for i in idx[half:]]
        assert not (set(tr) & set(te)), "train/test stems overlap"
        assert not (set(va) & set(te)), "val/test stems overlap"
        assert not (set(tr) & set(va)), "train/val stems overlap"
        n_tr, n_va, n_te = len(tr), len(va), len(te)

        g = torch.Generator()
        g.manual_seed(seed)
        dl_tr = DataLoader(PairedSet(cover_dir, stego_dir, tr, True),
                           batch_size=bs, shuffle=True, num_workers=2,
                           drop_last=True, generator=g)
        dl_va = DataLoader(PairedSet(cover_dir, stego_dir, va, False),
                           batch_size=bs, shuffle=False, num_workers=2)
        dl_te = DataLoader(PairedSet(cover_dir, stego_dir, te, False),
                           batch_size=bs, shuffle=False, num_workers=2)

        model = build_model().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr)
        lossf = nn.BCEWithLogitsLoss()
        use_amp = device.type == "cuda"
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        best_val, chosen, best_ep = 2.0, None, -1
        best_strict, since_better, ran = 2.0, 0, 0
        for ep in range(epochs):
            ran = ep + 1
            model.train()
            for x, y, _ in dl_tr:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    loss = lossf(model(x).squeeze(1), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

            # Epoch selection looks at VALIDATION only. The test set is
            # scored at the chosen epoch and never influences the choice.
            # `<=` keeps the later epoch on ties: the validation slice is
            # small, ties are common, and a longer-trained model is the safer
            # tie-break.
            vc, vs, _ = score_loader(model, dl_va, device, use_amp)
            val_pe = p_error(vc, vs)
            if val_pe <= best_val:
                best_val = val_pe
                chosen = score_loader(model, dl_te, device, use_amp)
                best_ep = ep
            # Patience counts STRICT improvements: with `<=` alone, a cell at
            # chance ties forever and would never stop.
            if val_pe < best_strict:
                best_strict, since_better = val_pe, 0
            else:
                since_better += 1
            if PATIENCE is not None and since_better >= PATIENCE:
                break

        sc, ss, keep = chosen
        test_pe = p_error(sc, ss)
        rev_pe = p_error(ss, sc)
        per_seed_pe.append(test_pe)
        per_seed_rev.append(rev_pe)
        per_seed_val.append(best_val)
        per_seed_md.append(md_at_fa(sc, ss))
        best_epochs.append(best_ep)
        epochs_run.append(ran)
        rows += [(seed, s, lbl, v) for s, lbl, v in keep]
        print(f"    seed {seed}: P_E {test_pe:.4f} "
              f"(val {best_val:.4f}, epoch {best_ep} of {ran} run)", flush=True)

    # Inversion guard. For a detectable cell with correctly oriented scores,
    # the forward P_E is low and the reversed one sits at ~0.5. If it is the
    # other way round the scores are upside down - and the forward number
    # would be reporting ~0.5, i.e. "secure", for a method that is not.
    #
    # Both conditions are needed. At chance level the two directions are just
    # noise around the same value, so "reversed < forward" alone fires at
    # random - including on the null test, where it would be a false alarm on
    # the one check that must be trusted. Requiring the reversed direction to
    # sit significantly BELOW chance means it fires only on a real signal
    # pointing the wrong way.
    chance, chance_sd = null_expectation(n_te)
    rev_mean, fwd_mean = float(np.mean(per_seed_rev)), float(np.mean(per_seed_pe))
    if rev_mean < chance - 3.0 * chance_sd and rev_mean < fwd_mean - 0.05:
        print("  !! SCORES LOOK INVERTED: reversed P_E "
              f"{np.mean(per_seed_rev):.4f} is lower than forward "
              f"{np.mean(per_seed_pe):.4f}. Higher score must mean 'stego'.",
              flush=True)

    out_scores.parent.mkdir(parents=True, exist_ok=True)
    with out_scores.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["seed", "image", "label", "score"])
        w.writerows(rows)

    return dict(
        p_e_mean=float(np.mean(per_seed_pe)),
        p_e_std=float(np.std(per_seed_pe)),
        p_e_per_seed=[float(v) for v in per_seed_pe],
        md_at_fa1_mean=float(np.mean(per_seed_md)),
        md_at_fa1_std=float(np.std(per_seed_md)),
        best_epoch_per_seed=best_epochs,
        epochs_run_per_seed=epochs_run,
        patience=PATIENCE,
        p_e_reversed_mean=float(np.mean(per_seed_rev)),
        val_p_e_per_seed=[float(v) for v in per_seed_val],
        # What P_E reads at this test size with NO signal. Not 0.50.
        p_e_chance_level=float(null_expectation(n_te)[0]),
        epoch_selection="validation",
        n_train_pairs=n_tr, n_val_pairs=n_va, n_test_pairs=n_te,
    )


def null_test(cover_dir, out, device, args) -> bool:
    """Covers labelled against other covers. Must return chance-level P_E.

    "Chance" is NOT 0.50 - see null_expectation. The gate is one-sided: a leak
    can only push P_E down, so we only test the lower bound.

    This is the check that catches label leakage, ordering artifacts and
    train/test contamination. Nothing downstream is trustworthy until it passes.
    """
    print("\nNULL TEST (cover vs cover) - expect chance-level P_E")
    stems = sorted((p.stem for p in cover_dir.glob("*.jpg")),
                   key=lambda s: (len(s), s))
    # The two groups MUST be drawn at random. BOSSbase is ordered by source
    # (camera), so "first half vs second half by file number" compares two
    # different kinds of image, and any competent detector separates them:
    # at 2,000 covers that read P_E 0.29 here and 0.31 under DCTR, with no leak
    # anywhere. Random groups of the same covers give 0.48.
    stems = [stems[i] for i in
             np.random.default_rng(20261002).permutation(len(stems))]
    half = len(stems) // 2
    a, b = stems[:half], stems[half:2 * half]

    # Build a fake 'stego' folder whose file N.jpg is a *different cover*, named
    # to pair with cover N.jpg. There is no embedded signal to find.
    with tempfile.TemporaryDirectory() as td:
        fake = pathlib.Path(td)
        for sa, sb in zip(a, b):
            shutil.copyfile(cover_dir / f"{sb}.jpg", fake / f"{sa}.jpg")
        r = run_cell(cover_dir, fake, a, device, max(5, args.epochs // 4),
                     args.batch_size, args.lr, out / "scores" / "NULL_TEST.csv")

    mean, sd = null_expectation(r["n_test_pairs"])
    # Seeds reuse the same images under different splits, so they are not
    # independent draws; 2.5 sd of a SINGLE seed is a deliberately loose floor.
    # A real leak (the network recognising photographs) lands far below it.
    floor = mean - 2.5 * sd
    r["null_expected"], r["null_floor"] = mean, floor
    print(f"\n  NULL TEST P_E = {r['p_e_mean']:.4f}   "
          f"(chance at {r['n_test_pairs']} test pairs = {mean:.4f}, "
          f"pass if >= {floor:.4f})")
    ok = r["p_e_mean"] >= floor
    print("  " + ("PASS - pipeline finds no signal where none exists" if ok else
                  "FAIL - PIPELINE IS LEAKING. Stop and debug (02_PITFALLS.md 2)"))
    (out / "null_test.json").write_text(json.dumps(r, indent=2), encoding="utf-8")
    return ok


def resolve_data_root(given: pathlib.Path) -> pathlib.Path:
    """Find the folder that actually contains MANIFEST.json.

    Kaggle mounts a dataset at /kaggle/input/<slug>/, and because our zip has a
    top-level `dataset/` folder the real root is usually
    /kaggle/input/<slug>/dataset. Rather than making anyone get that right, we
    search: the given path, then one or two levels down.
    """
    if (given / "MANIFEST.json").exists():
        return given
    for cand in sorted(given.glob("*/MANIFEST.json")) + sorted(given.glob("*/*/MANIFEST.json")):
        print(f"note: using data root {cand.parent} (MANIFEST.json found there)")
        return cand.parent
    raise SystemExit(
        "No MANIFEST.json under " + str(given) + "\n"
        "Point --data at the folder containing MANIFEST.json. On Kaggle that is\n"
        "usually /kaggle/input/<dataset-slug>/dataset\n"
        "Tip: run  !find /kaggle/input -name MANIFEST.json  to locate it.")


# ------------------------------------------------------------------- main ---
def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="dataset root (has MANIFEST.json)")
    ap.add_argument("--out", required=True, help="output dir for m4_results")
    ap.add_argument("--cells", nargs="*", default=None, help="specific cell names")
    ap.add_argument("--all", action="store_true", help="run every cell")
    ap.add_argument("--null-test", action="store_true", help="run the null test first")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--time-budget-min", type=float, default=None,
                    help="stop cleanly before starting a cell that would not "
                         "finish inside this many minutes from now")
    ap.add_argument("--patience", type=int, default=None,
                    help="early-stop a seed after N epochs without a strict "
                         "validation improvement")
    ap.add_argument("--seeds", type=int, default=5,
                    help="number of seeds (0..N-1); 5 for the 300-image grid")
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    args = ap.parse_args()

    global SEEDS, PATIENCE
    SEEDS = list(range(args.seeds))
    PATIENCE = args.patience

    data = resolve_data_root(pathlib.Path(args.data))
    out = pathlib.Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")
    if device.type != "cuda":
        print("WARNING: no GPU. This will take days on CPU.")

    manifest = json.loads((data / "MANIFEST.json").read_text(encoding="utf-8"))
    cover_dir = data / manifest["cover_dir"]

    if args.null_test:
        if not null_test(cover_dir, out, device, args):
            return 1
        if not (args.all or args.cells):
            return 0

    cells = manifest["cells"]
    if args.cells:
        cells = [c for c in cells if c["cell"] in args.cells]
        missing = set(args.cells) - {c["cell"] for c in cells}
        if missing:
            print(f"unknown cell names: {sorted(missing)}")
            return 1
    elif not args.all:
        print("nothing to do: pass --all or --cells NAME")
        return 0

    res_path = out / "results.json"
    results = (json.loads(res_path.read_text(encoding="utf-8"))
               if res_path.exists() else [])
    done = {(r["cell"], r["detector"]) for r in results}

    t_begin, longest = time.time(), 0.0
    for c in cells:
        if (c["cell"], "EfficientNet-B0") in done:
            print(f"skip {c['cell']} (already in results.json)")
            continue
        # A Kaggle session that hits its 12-hour cap is killed and its output
        # is lost. Stopping one cell early keeps everything finished so far;
        # the remaining cells are picked up by a rerun.
        if args.time_budget_min is not None:
            used = (time.time() - t_begin) / 60
            if used + longest * 1.15 > args.time_budget_min:
                print(f"TIME BUDGET: {used:.0f} min used, next cell needs about "
                      f"{longest * 1.15:.0f}; stopping cleanly before {c['cell']}")
                break
        stego_dir = data / c["stego_dir"]
        stems = sorted((p.stem for p in stego_dir.glob("*.jpg")),
                       key=lambda s: (len(s), s))
        # Exact pairing only. SCRIPT cells ship 299 stegos against 300 covers;
        # dropping the unmatched cover keeps the pairing exact. Never pad or
        # duplicate to make the counts line up.
        stems = [s for s in stems if (cover_dir / f"{s}.jpg").exists()]
        print(f"\n{c['cell']}  ({len(stems)} pairs)", flush=True)

        t0 = time.time()
        try:
            r = run_cell(cover_dir, stego_dir, stems, device, args.epochs,
                         args.batch_size, args.lr,
                         out / "scores" / f"{c['cell']}.csv")
            entry = dict(cell=c["cell"], method=c["method"],
                         payload_bpnzac=c["payload_bpnzac"], arm=c["arm"],
                         detector="EfficientNet-B0", status="ok",
                         seeds=SEEDS, epochs=args.epochs,
                         train_minutes=round((time.time() - t0) / 60, 1), **r)
            longest = max(longest, (time.time() - t0) / 60)
            print(f"  -> P_E {entry['p_e_mean']:.4f} +/- {entry['p_e_std']:.4f}"
                  f"   (chance = {entry['p_e_chance_level']:.4f})")
        except Exception as exc:  # noqa: BLE001
            # Record failures as failures. Never substitute a default value -
            # that is exactly the defect we found in four published drivers.
            entry = dict(cell=c["cell"], method=c["method"],
                         payload_bpnzac=c["payload_bpnzac"], arm=c["arm"],
                         detector="EfficientNet-B0", status="failed",
                         error=str(exc)[:300])
            print(f"  FAILED: {exc}")

        results.append(entry)
        # Checkpoint after every cell so a session timeout costs one cell, not all.
        res_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    print(f"\nwrote {res_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile dctr.py
"""DCTR features (Holub & Fridrich, 8000-dim) - a Python port of DCTR.m.

Ported line for line from the DDE Lab reference
(`baselines/features/DCTR_matlab_v1.1/matlab/DCTR.m`) so the extraction can run
where MATLAB cannot. A feature extractor that is subtly wrong is very hard to
notice and poisons every P_E built on it, so this port is only to be trusted
because `experiments/exp15_dctr_port_check.py` shows it reproduces the MATLAB
features on images we already extracted with the reference code.

Three places where a naive port goes wrong, all handled below:
  * MATLAB `round` rounds half away from zero; numpy/torch round half to even.
  * `conv2(..., 'valid')` is a true convolution - the kernel is flipped.
  * MATLAB indexes from 1, so shift (i, j) is offset (i-1, j-1) here.

Runs on CPU or GPU; float64 throughout, because the reference is double.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F

T = 4
DIM = 8000


def quant_step(qf: int) -> float:
    if qf < 50:
        return min(8.0 * (50.0 / qf), 100.0)
    return max(8.0 * (2.0 - (qf / 50.0)), 0.2)


def _dct_matrix() -> np.ndarray:
    """A[x, u] = 0.5 * c(u) * cos((2x+1) u pi / 16): the matrix DCTR.m calls A."""
    k, l = np.meshgrid(np.arange(8), np.arange(8))
    a = 0.5 * np.cos(((2.0 * k + 1.0) * l * math.pi) / 16.0)
    a[0, :] = a[0, :] / math.sqrt(2.0)
    return a.T


def _merged_coordinates() -> list[list[tuple[int, int]]]:
    """The 25 groups of symmetric phase shifts, as 0-based offsets."""
    out = []
    for i in range(1, 6):
        for j in range(1, 6):
            cand = {(i, j), (i, 10 - j), (10 - i, j), (10 - i, 10 - j)}
            out.append(sorted((r - 1, c - 1) for r, c in cand if r < 9 and c < 9))
    return out


class DCTR:
    def __init__(self, qf: int, device: str | torch.device = "cpu"):
        self.device = torch.device(device)
        self.q = quant_step(qf)
        a = _dct_matrix()
        self.A = torch.tensor(a, dtype=torch.float64, device=self.device)
        # conv2 flips the kernel: tap a of the image meets element 7-a.
        self.Af = torch.flip(self.A, dims=[0])
        self._row_offset = (torch.arange(64 * 64, dtype=torch.int32,
                                         device=self.device)[:, None] * (T + 2))
        # 64 bases in DCTR.m's order: modeIndex = (mode_r-1)*8 + mode_c.
        bases = np.stack([np.outer(a[:, r], a[:, c])
                          for r in range(8) for c in range(8)])
        # conv2 'valid' flips the kernel; F.conv2d correlates. Flip to match.
        k = torch.tensor(bases[:, ::-1, ::-1].copy(), dtype=torch.float64,
                         device=self.device)
        self.kernels = k.unsqueeze(1)                       # (64, 1, 8, 8)
        # group[r, c] = which of the 25 merged groups phase (r, c) feeds.
        g = torch.zeros(8, 8, 25, dtype=torch.float64, device=self.device)
        for m, coords in enumerate(_merged_coordinates()):
            for r, c in coords:
                g[r, c, m] = 1.0
        self.group = g.reshape(64, 25)

    def spatial(self, coef_blocks: np.ndarray, qt: np.ndarray) -> torch.Tensor:
        """Decompress without rounding or clipping, minus the +128 offset.

        coef_blocks: (bh, bw, 8, 8) quantised DCT coefficients.
        Returns I_spatial - 128, which is what DCTR.m convolves.
        """
        d = torch.tensor(np.asarray(coef_blocks, dtype=np.float64)
                         * np.asarray(qt, dtype=np.float64),
                         device=self.device)
        x = self.A @ d @ self.A.T                           # blockwise idct2
        bh, bw = x.shape[:2]
        return x.permute(0, 2, 1, 3).reshape(bh * 8, bw * 8)

    @torch.no_grad()
    def features(self, coef_blocks: np.ndarray, qt: np.ndarray) -> np.ndarray:
        img = self.spatial(coef_blocks, qt)
        # The 64 DCT bases are separable (base = A[:, r] outer A[:, c]), so the
        # 2-D convolution is two 1-D ones: along y with the 8 row vectors, then
        # along x with the 8 column vectors. Same mathematics as
        # conv2(I, base, 'valid') at about a seventh of the arithmetic, and the
        # result comes out as (y, x, r, c) - already in DCTR.m's mode order
        # r*8 + c, with no copy needed. The summation ORDER differs from a
        # direct 2-D convolution, so a value landing exactly on a rounding
        # boundary can fall the other way; exp15 re-checks P_E against the
        # MATLAB reference for this reason.
        t1 = torch.tensordot(img.unfold(0, 8, 1), self.Af, dims=([2], [0]))  # (H-7, W, r)
        r = torch.tensordot(t1.unfold(1, 8, 1), self.Af, dims=([3], [0]))    # (H-7, W-7, r, c)
        h, w = r.shape[:2]

        # abs(round(R/q)) then threshold, in place and straight to uint8: the
        # same operations in the same order as floor(|R| / q + 0.5),
        # clamp(max=T), without four temporaries.
        r = r.abs_().div_(self.q).add_(0.5).floor_().clamp_(max=T).to(torch.uint8)

        # counts[mode, phase(8x8), value]: the histogram of
        # R(r_shift:8:end, c_shift:8:end) for all 64 phases at once. Pad to a
        # multiple of 8 with a sentinel no bin matches, regroup so each row is
        # one (mode, phase), and histogram all rows with a single bincount.
        # Integer counting: exactly what slicing phase by phase gives.
        ph, pw = -h % 8, -w % 8
        rp = F.pad(r.view(h, w, 64), (0, 0, 0, pw, 0, ph), value=T + 1)
        x = (rp.view((h + ph) // 8, 8, (w + pw) // 8, 8, 64)
               .permute(4, 1, 3, 0, 2).reshape(64 * 64, -1))
        key = x.to(torch.int32) + self._row_offset
        counts = (torch.bincount(key.view(-1), minlength=64 * 64 * (T + 2))
                  .view(64 * 64, T + 2)[:, :T + 1].reshape(64, 64, T + 1).double())
        merged = torch.einsum("mpv,pg->mgv", counts, self.group)   # (64, 25, 5)
        merged = merged / merged.sum(-1, keepdim=True)
        return merged.reshape(-1).cpu().numpy()


def read_jpeg(path) -> tuple[np.ndarray, np.ndarray]:
    """(coefficient blocks, quant table) of the luminance channel."""
    import jpeglib

    im = jpeglib.read_dct(str(path))
    return np.asarray(im.Y), np.asarray(im.qt[0])


def extract(paths, qf: int, device="cpu", progress: int = 0) -> np.ndarray:
    ex = DCTR(qf, device)
    out = np.empty((len(paths), DIM), dtype=np.float64)
    for i, p in enumerate(paths):
        out[i] = ex.features(*read_jpeg(p))
        if progress and (i + 1) % progress == 0:
            print(f"    {i + 1}/{len(paths)}", flush=True)
    return out

In [ ]:
%%writefile ensemble.py
"""FLD ensemble classifier for steganalysis (Kodovsky, Fridrich & Holub, 2011).

The standard classifier for high-dimensional handcrafted steganalysis features.
A single FLD on 8000 DCTR dimensions with only a few thousand training examples
would be hopelessly overdetermined; the ensemble instead trains many FLDs on
random low-dimensional subspaces and votes.

Implemented on scikit-learn's LDA rather than the DDE lab's MATLAB, because
MATLAB's Statistics and Machine Learning Toolbox is not installed on this
machine and sklearn is (measured: 0.30 s per base learner at d_sub=800,
n=4000 — ~2.5 min for 500 learners).

Deviation worth stating in the paper: the reference implementation selects
``d_sub`` and ``L`` automatically by out-of-bag error. Here they are explicit
arguments with the reference defaults, and :func:`search_d_sub` reproduces the
OOB search when it matters.
"""
from __future__ import annotations

import dataclasses
from typing import Sequence

import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

__all__ = ["EnsembleFLD", "oob_error", "search_d_sub"]


@dataclasses.dataclass
class EnsembleFLD:
    """Ensemble of FLDs on random feature subspaces.

    Attributes
    ----------
    d_sub:
        Dimensionality of each random subspace.
    n_estimators:
        Number of base learners (``L`` in the paper).
    seed:
        RNG seed; fixed so a run reproduces.
    """

    d_sub: int = 800
    n_estimators: int = 500
    seed: int = 0

    def fit(self, X: np.ndarray, y: np.ndarray) -> "EnsembleFLD":
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y).ravel()
        n_features = X.shape[1]
        d = min(self.d_sub, n_features)

        rng = np.random.default_rng(self.seed)
        self._subspaces: list[np.ndarray] = []
        self._learners: list[LDA] = []
        self._bags: list[np.ndarray] = []
        n = X.shape[0]

        for _ in range(self.n_estimators):
            cols = rng.choice(n_features, d, replace=False)
            # Bootstrap so out-of-bag error is available, as in the reference.
            bag = rng.choice(n, n, replace=True)
            # shrinkage='auto' (Ledoit-Wolf) stands in for the reference's
            # regularised within-class scatter; without it the covariance is
            # singular whenever d_sub approaches the bag size.
            clf = LDA(solver="lsqr", shrinkage="auto")
            clf.fit(X[np.ix_(bag, cols)], y[bag])
            self._subspaces.append(cols)
            self._learners.append(clf)
            self._bags.append(bag)

        self._n_train = n
        return self

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        """Mean vote margin. Larger means more likely stego (class 1)."""
        X = np.asarray(X, dtype=np.float64)
        votes = np.zeros(X.shape[0], dtype=np.float64)
        for cols, clf in zip(self._subspaces, self._learners):
            votes += clf.predict(X[:, cols]).astype(np.float64)
        return votes / len(self._learners)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self.decision_function(X) >= 0.5).astype(int)


def oob_error(model: EnsembleFLD, X: np.ndarray, y: np.ndarray) -> float:
    """Out-of-bag error — the reference's criterion for choosing d_sub."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y).ravel()
    n = X.shape[0]
    votes = np.zeros(n)
    counts = np.zeros(n)
    for cols, clf, bag in zip(model._subspaces, model._learners, model._bags):
        oob = np.setdiff1d(np.arange(n), bag, assume_unique=False)
        if oob.size == 0:
            continue
        votes[oob] += clf.predict(X[np.ix_(oob, cols)])
        counts[oob] += 1
    seen = counts > 0
    pred = (votes[seen] / counts[seen] >= 0.5).astype(int)
    return float(np.mean(pred != y[seen]))


def search_d_sub(X: np.ndarray, y: np.ndarray,
                 candidates: Sequence[int] = (200, 400, 800, 1600),
                 n_estimators: int = 100, seed: int = 0) -> tuple[int, dict]:
    """Pick d_sub by out-of-bag error, as the reference implementation does.

    Uses fewer base learners than the final model — the OOB curve's *shape* is
    what matters here, not its absolute level.
    """
    scores = {}
    for d in candidates:
        m = EnsembleFLD(d_sub=d, n_estimators=n_estimators, seed=seed).fit(X, y)
        scores[d] = oob_error(m, X, y)
    best = min(scores, key=scores.get)
    return best, scores

In [ ]:
%%writefile dctr_kaggle.py
"""DCTR + FLD-ensemble steganalysis over a stego-eval dataset.

Part A of the scale notebook: the feature-based detector, run where MATLAB is
not available. Uses the Python DCTR port (dctr.py), which is validated against
the DDE Lab MATLAB reference by experiments/exp15_dctr_port_check.py.

Cover and stego features are ALWAYS extracted by the same code here. Never
combine these with MATLAB-extracted features in one comparison: the two break
rounding ties differently, and that difference alone is detectable.

    python dctr_kaggle.py --data /kaggle/input/.../dataset --out /kaggle/working/dctr
"""
from __future__ import annotations

import argparse
import csv
import json
import pathlib
import time

import numpy as np
import torch

from dctr import DCTR, DIM, read_jpeg
from ensemble import EnsembleFLD


def numeric(stems):
    return sorted(stems, key=lambda s: (len(s), s))


def p_error(sc: np.ndarray, ss: np.ndarray) -> float:
    """min over thresholds of 0.5*(P_FA + P_MD); higher score = more stego."""
    best = 1.0
    for t in np.unique(np.concatenate([sc, ss])):
        best = min(best, 0.5 * ((sc >= t).mean() + (ss < t).mean()))
    return float(best)


def md_at_fa(sc, ss, target_fa=0.01) -> float:
    return float((ss < np.quantile(sc, 1 - target_fa)).mean())


def extract_dir(ex: DCTR, folder: pathlib.Path, stems: list[str],
                cache: pathlib.Path) -> np.ndarray:
    """Features for folder/<stem>.jpg in the given order, cached as float32.

    Reads are deliberately sequential. jpeglib is NOT thread-safe: reading
    from a thread pool kills the interpreter outright, with no traceback and
    no Python exception to catch. Measured, not assumed - do not parallelise
    this with threads.
    """
    meta = cache.with_suffix(".stems.json")
    if cache.exists() and meta.exists() and json.loads(meta.read_text()) == stems:
        return np.load(cache).astype(np.float64)
    out = np.empty((len(stems), DIM), dtype=np.float32)
    t0 = time.time()
    for i, s in enumerate(stems):
        out[i] = ex.features(*read_jpeg(folder / f"{s}.jpg"))
        if (i + 1) % 500 == 0:
            print(f"    {i + 1}/{len(stems)}  "
                  f"({(time.time() - t0) / (i + 1) * 1000:.0f} ms/image)", flush=True)
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache, out)
    meta.write_text(json.dumps(stems))
    return out.astype(np.float64)


def run_cell(Xc, Xs, stems, seeds, d_sub, n_est):
    """Paired 50/50 split per seed. Returns (summary dict, score rows)."""
    pes, revs, mds, rows = [], [], [], []
    n = len(stems)
    for seed in range(seeds):
        idx = np.random.default_rng(seed).permutation(n)
        tr, te = idx[:n // 2], idx[n // 2:]
        X = np.vstack([Xc[tr], Xs[tr]])
        y = np.concatenate([np.zeros(len(tr)), np.ones(len(tr))])
        m = EnsembleFLD(d_sub=d_sub, n_estimators=n_est, seed=seed).fit(X, y)
        sc, ss = m.decision_function(Xc[te]), m.decision_function(Xs[te])
        pes.append(p_error(sc, ss))
        revs.append(p_error(ss, sc))
        mds.append(md_at_fa(sc, ss))
        rows += [(seed, stems[i], "cover", float(v)) for i, v in zip(te, sc)]
        rows += [(seed, stems[i], "stego", float(v)) for i, v in zip(te, ss)]
        print(f"    seed {seed}: P_E {pes[-1]:.4f}", flush=True)
    return dict(p_e_mean=float(np.mean(pes)), p_e_std=float(np.std(pes)),
                p_e_per_seed=[float(v) for v in pes],
                p_e_reversed_mean=float(np.mean(revs)),
                md_at_fa1_mean=float(np.mean(mds)),
                md_at_fa1_std=float(np.std(mds)),
                n_train_pairs=n // 2, n_test_pairs=n - n // 2,
                seeds=list(range(seeds))), rows


def write_scores(path: pathlib.Path, rows) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["seed", "image", "label", "score"])
        w.writerows(rows)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--device", default="cuda:0" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--d-sub", type=int, default=800)
    ap.add_argument("--n-est", type=int, default=200)
    ap.add_argument("--limit", type=int, default=None, help="first N covers (testing)")
    ap.add_argument("--cells", nargs="*", default=None)
    ap.add_argument("--null-only", action="store_true",
                    help="cover features and the null test, no cells")
    args = ap.parse_args()

    data, out = pathlib.Path(args.data), pathlib.Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    mf = json.loads((data / "MANIFEST.json").read_text(encoding="utf-8"))
    qf = int(mf.get("image_spec", {}).get("cover_quality_factor", 65))
    cover_dir = data / mf["cover_dir"]
    cov_stems = numeric(p.stem for p in cover_dir.glob("*.jpg"))
    if args.limit:
        cov_stems = cov_stems[:args.limit]
    print(f"device {args.device} | QF {qf} | {len(cov_stems)} covers", flush=True)

    ex = DCTR(qf, args.device)
    feat = out / "features"
    print("extracting cover features...", flush=True)
    Xcov = extract_dir(ex, cover_dir, cov_stems, feat / "cover.npy")
    pos = {s: i for i, s in enumerate(cov_stems)}

    # Null test: covers against OTHER covers. With ensemble vote counts the
    # scores have many ties, so chance is not the continuous-score value; this
    # measured number is the chance level for every DCTR result below.
    null_path = out / "null_test.json"
    if not null_path.exists():
        # Random groups, never "first half vs second half": BOSSbase is
        # ordered by source, and splitting by file number makes the two groups
        # genuinely different (measured: P_E 0.31 that way, 0.48 at random).
        perm = np.random.default_rng(20261002).permutation(len(cov_stems))
        half = len(cov_stems) // 2
        ia, ib = perm[:half], perm[half:2 * half]
        print("\nNULL TEST (cover vs cover, random groups)", flush=True)
        r, rows = run_cell(Xcov[ia], Xcov[ib], [cov_stems[i] for i in ia],
                           args.seeds, args.d_sub, args.n_est)
        r["grouping"] = "random"
        write_scores(out / "scores" / "NULL_TEST.csv", rows)
        null_path.write_text(json.dumps(r, indent=2), encoding="utf-8")
        print(f"  NULL TEST P_E = {r['p_e_mean']:.4f} +/- {r['p_e_std']:.4f} "
              f"at {r['n_test_pairs']} test pairs")
    null = json.loads(null_path.read_text())
    if args.null_only:
        return 0

    res_path = out / "results.json"
    results = json.loads(res_path.read_text()) if res_path.exists() else []
    done = {r["cell"] for r in results}
    cells = [c for c in mf["cells"] if not args.cells or c["cell"] in args.cells]

    for c in cells:
        if c["cell"] in done:
            print(f"skip {c['cell']} (already in results.json)")
            continue
        sdir = data / c["stego_dir"]
        stems = [s for s in numeric(p.stem for p in sdir.glob("*.jpg")) if s in pos]
        print(f"\n{c['cell']}  ({len(stems)} pairs)", flush=True)
        t0 = time.time()
        try:
            Xs = extract_dir(ex, sdir, stems, feat / f"{c['cell']}.npy")
            Xc = Xcov[[pos[s] for s in stems]]
            r, rows = run_cell(Xc, Xs, stems, args.seeds, args.d_sub, args.n_est)
            write_scores(out / "scores" / f"{c['cell']}.csv", rows)
            entry = dict(cell=c["cell"], method=c["method"],
                         payload_bpnzac=c["payload_bpnzac"], arm=c["arm"],
                         detector="DCTR+FLD", status="ok",
                         p_e_chance_level=null["p_e_mean"],
                         minutes=round((time.time() - t0) / 60, 1), **r)
            print(f"  -> P_E {r['p_e_mean']:.4f} +/- {r['p_e_std']:.4f}   "
                  f"(null {null['p_e_mean']:.4f})")
        except Exception as exc:  # noqa: BLE001 - record, never substitute
            entry = dict(cell=c["cell"], method=c["method"],
                         payload_bpnzac=c["payload_bpnzac"], arm=c["arm"],
                         detector="DCTR+FLD", status="failed", error=str(exc)[:300])
            print(f"  FAILED: {exc}")
        results.append(entry)
        res_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    print(f"\nwrote {res_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# ---- Cell 2: Part A - DCTR + FLD ---------------------------------------
# Features for covers and all cells on GPU 0, then an FLD ensemble with five
# paired 50/50 splits per cell. Includes its own cover-vs-cover null test.
# A failure here is reported loudly but does not stop Part B.
#
# RUN_PART_A = False skips it. Version 1 of this notebook already produced the
# DCTR results for the 2,000-image dataset (3.3 h), so a rerun for the CNN
# should not pay for them again. Set it to True for a new dataset.
RUN_PART_A = False
PART_A_OK = False
if RUN_PART_A:
    p = subprocess.run([sys.executable, "dctr_kaggle.py", "--data", str(DATA),
                        "--out", str(OUT / "dctr"), "--device", "cuda:0"])
    PART_A_OK = p.returncode == 0
    print()
    print("PART A " + ("finished" if PART_A_OK else
                       "FAILED (exit " + str(p.returncode) + ") - continuing to Part B"))
else:
    print("PART A skipped (RUN_PART_A = False)")
print("elapsed: %.0f min" % ((time.time() - NOTEBOOK_T0) / 60))

In [ ]:
# ---- Cell 3: Part B - CNN null test (gate) -----------------------------
# Covers against other covers. The script prints the chance level for its
# test size and the pass floor, and exits non-zero below the floor - in which
# case this cell raises and the CNN grid does not run.
CNN = OUT / "cnn"
SEEDS, EPOCHS, BATCH, PATIENCE = "3", "40", "16", ""
CNN_CELLS = ['SCRIPT-R0_0.05', 'SCRIPT-R0_0.2', 'SCRIPT-R0_0.1']          # empty = every cell in the manifest
COMMON = ["--data", str(DATA), "--seeds", SEEDS, "--epochs", EPOCHS,
          "--batch-size", BATCH]
if PATIENCE:
    COMMON += ["--patience", PATIENCE]
print("CNN settings: seeds", SEEDS, "| max epochs", EPOCHS, "| batch", BATCH,
      "| patience", PATIENCE or "none", "|", len(CNN_CELLS) or "all", "cells")

p = subprocess.run([sys.executable, "m4.py", *COMMON, "--out", str(CNN),
                    "--null-test"])
if p.returncode != 0:
    raise RuntimeError("CNN null test failed - the pipeline is leaking. "
                       "Do not run the grid; read the output above.")
print("elapsed: %.0f min" % ((time.time() - NOTEBOOK_T0) / 60))

In [ ]:
# ---- Cell 4: Part B - CNN grid, one process per GPU --------------------
# Cells are dealt alternately to the GPUs. Each process checkpoints after
# every cell and stops cleanly if the next cell would not fit in the time left.
import os

names = list(CNN_CELLS) or [c["cell"] for c in mf["cells"]]
unknown = set(names) - {c["cell"] for c in mf["cells"]}
if unknown:
    raise SystemExit("cells not in the manifest: " + ", ".join(sorted(unknown)))
parts = [names[i::N_GPU] for i in range(N_GPU)]
budget = SESSION_LIMIT_MIN - (time.time() - NOTEBOOK_T0) / 60 - 15
print("time budget for the CNN grid: %.0f min" % budget)

procs = []
for g, part in enumerate(parts):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = open(OUT / ("cnn_gpu%d.log" % g), "w")
    cmd = [sys.executable, "-u", "m4.py", *COMMON, "--out", str(OUT / ("cnn_part%d" % g)),
           "--cells", *part, "--time-budget-min", "%.0f" % budget]
    print("GPU", g, "->", len(part), "cells:", ", ".join(part))
    procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log))

# Echo progress lines while waiting, so the log shows the run is alive.
seen = [0] * len(procs)
while any(pr.poll() is None for pr, _ in procs):
    time.sleep(60)
    for g in range(len(procs)):
        lines = (OUT / ("cnn_gpu%d.log" % g)).read_text(errors="replace").splitlines()
        for ln in lines[seen[g]:]:
            if "Warning" in ln or "warn" in ln or "autocast" in ln or "GradScaler" in ln:
                continue
            print("[gpu%d] %s" % (g, ln), flush=True)
        seen[g] = len(lines)
for g, (pr, log) in enumerate(procs):
    log.close()
    print("GPU", g, "process exit code", pr.returncode)
print("elapsed: %.0f min" % ((time.time() - NOTEBOOK_T0) / 60))

In [ ]:
# ---- Cell 5: merge, compare, package -----------------------------------
import shutil

# Merge the per-GPU CNN outputs into one results.json and one scores folder.
cnn_res = []
(CNN / "scores").mkdir(parents=True, exist_ok=True)
for g in range(N_GPU):
    part = OUT / ("cnn_part%d" % g)
    if (part / "results.json").exists():
        cnn_res += json.loads((part / "results.json").read_text())
    if (part / "scores").is_dir():
        for f in (part / "scores").glob("*.csv"):
            shutil.copy2(f, CNN / "scores" / f.name)
(CNN / "results.json").write_text(json.dumps(cnn_res, indent=2))

dctr_res = []
if (OUT / "dctr" / "results.json").exists():
    dctr_res = json.loads((OUT / "dctr" / "results.json").read_text())
D = {r["cell"]: r for r in dctr_res if r["status"] == "ok"}
C = {r["cell"]: r for r in cnn_res if r["status"] == "ok"}

def null_of(folder):
    f = OUT / folder / "null_test.json"
    return json.loads(f.read_text()) if f.exists() else None

nd, nc = null_of("dctr"), null_of("cnn")
print("null tests (cover vs cover):")
if nd:
    print("  DCTR+FLD  P_E %.4f at %d test pairs" % (nd["p_e_mean"], nd["n_test_pairs"]))
if nc:
    print("  CNN       P_E %.4f at %d test pairs (chance %.4f, floor %.4f)"
          % (nc["p_e_mean"], nc["n_test_pairs"], nc["null_expected"], nc["null_floor"]))
print()
print("cell".ljust(18) + "DCTR+FLD".rjust(10) + "+/-".rjust(8)
      + "CNN".rjust(10) + "+/-".rjust(8) + "  lower")
print("-" * 62)
for c in mf["cells"]:
    n = c["cell"]
    d, k = D.get(n), C.get(n)
    line = n.ljust(18)
    line += ("%.4f" % d["p_e_mean"]).rjust(10) + ("%.4f" % d["p_e_std"]).rjust(8) if d else "-".rjust(10) + "-".rjust(8)
    line += ("%.4f" % k["p_e_mean"]).rjust(10) + ("%.4f" % k["p_e_std"]).rjust(8) if k else "-".rjust(10) + "-".rjust(8)
    if d and k:
        line += "  " + ("CNN" if k["p_e_mean"] < d["p_e_mean"] - 0.005 else
                        "DCTR" if d["p_e_mean"] < k["p_e_mean"] - 0.005 else "tie")
    print(line)

missing_cnn = [c["cell"] for c in mf["cells"] if c["cell"] not in C]
missing_dctr = [c["cell"] for c in mf["cells"] if c["cell"] not in D]
print()
print("DCTR cells done: %d/%d" % (len(D), len(mf["cells"])))
print("CNN  cells done: %d/%d" % (len(C), len(mf["cells"])))
if missing_cnn:
    print("CNN cells NOT run (time budget or failure):", ", ".join(missing_cnn))
if missing_dctr:
    print("DCTR cells NOT run:", ", ".join(missing_dctr))

# Inverted-score check for the CNN (inverted scores read as ~0.5, 'secure').
for n, r in C.items():
    if (r["p_e_reversed_mean"] < r["p_e_chance_level"] - 0.06
            and r["p_e_reversed_mean"] < r["p_e_mean"] - 0.05):
        print("  CHECK: " + n + " CNN scores look INVERTED")

notes = [
    "# Scale run notes", "",
    "- Dataset: %d covers, %d cells" % (N_COVERS, len(mf["cells"])),
    "- Hardware: %d x %s" % (N_GPU, torch.cuda.get_device_name(0)),
    "- torch %s" % torch.__version__,
    "- Part A: DCTR (Python port, validated against the MATLAB reference) + FLD "
    "ensemble, d_sub 800, L 200, 5 paired 50/50 splits",
    "- Part B: EfficientNet-B0, ImageNet-pretrained, 1 channel, 512x512, no resize; "
    "%s seeds, %s epochs, batch %s, AdamW lr 1e-4, AMP; paired 40/10/50 split, "
    "epoch chosen on validation" % (SEEDS, EPOCHS, BATCH),
    "- DCTR null P_E: %s" % ("%.4f" % nd["p_e_mean"] if nd else "NOT RUN"),
    "- CNN null P_E: %s" % ("%.4f" % nc["p_e_mean"] if nc else "NOT RUN"),
    "- DCTR cells done: %d/%d; CNN cells done: %d/%d"
    % (len(D), len(mf["cells"]), len(C), len(mf["cells"])),
    "- Total minutes: %.0f" % ((time.time() - NOTEBOOK_T0) / 60),
]
(OUT / "NOTES.md").write_text(chr(10).join(notes) + chr(10))

# The zip holds results, scores, logs and notes. The DCTR feature matrices
# (about 1.2 GB) stay as separate output files: download them only if needed.
stage = pathlib.Path("/kaggle/working/_zip")
if stage.exists():
    shutil.rmtree(stage)
for sub in ("dctr", "cnn"):
    if (OUT / sub).is_dir():
        shutil.copytree(OUT / sub, stage / sub,
                        ignore=shutil.ignore_patterns("features"))
for f in OUT.glob("*.log"):
    shutil.copy2(f, stage / f.name)
shutil.copy2(OUT / "NOTES.md", stage / "NOTES.md")
shutil.make_archive("/kaggle/working/scale_results", "zip", stage)
shutil.rmtree(stage)
print()
print("wrote /kaggle/working/scale_results.zip - download it from the Output tab")